#### TRANSFORM `PAYMENTS`
1. EXTRACT DATE & TIME FROM TIMESTAMP AND CREATE NEW COLUMNS PAYMENT_DATE & PAYMENT_TIME
2. MAP `PAYMENT_STATUS` TO CONTAIN DESCRIPTIVE VALUES
- 1 - Success
- 2 - Pending
- 3 - Cancelled
- 4 - Failed
3. WRITE TRANSFORMED DATA TO SILVER SCHEMA

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, col, when, date_format

In [0]:
payments_df = spark.table('''GIZMO.BRONZE.PAYMENTS_EXTERNAL''')
payments_select_df = payments_df.select(
                                      'customer_id', 
                                      'payment_id', 
                                      'payment_date',
                                      to_date(col("payment_date"), "yyyy-MM-dd").alias('date_of_payment'),
                                      'payment_status',
                                      date_format(col("payment_date"), "HH:mm:ss").alias('time_of_payment'),
                                      ).withColumn(
                                          'payment_category', 
                                          when(col("payment_status") == 1, "Success")
                                          .when(col("payment_status") == 2, "Pending")
                                          .when(col("payment_status") == 3, "Cancelled")
                                          .when(col("payment_status") == 4, "Failed")
                                          .otherwise("Unknown")
                                      ).drop('payment_status')

display(payments_select_df.limit(10))

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: PAYMENTS

In [0]:
payments_select_df.write.mode('overwrite').format('delta').saveAsTable('gizmo.silver.payments_delta')

#### VALIDATE AND QUERY `GIZMO.SILVER.PAYMENTS_DELTA`

In [0]:
%sql
SELECT * from GIZMO.SILVER.PAYMENTS_SILVER;

In [0]:
%python
dbutils.notebook.exit("PAYMENTS LOADED INTO GIZMO.SILVER.PAYMENTS")